# E3: semillas 6 a 10 de los rivales, sesión A

Mismo cuaderno, mismo paquete y misma receta que el benchmark del E2. Cambian la carpeta de Drive
(`tesis_imo_e3_semillas_v1`) y las semillas (89, 97, 101, 103, 107).


Redes de esta sesión: transunet_2d, pidnet_s. Las otras sesiones corren las demás; pueden ir en paralelo y escriben
en subcarpetas distintas (`sesion_A`, `sesion_B`, `sesion_C`), porque cada una guarda su propio `preflight.json`.

1. Entorno de ejecución → Cambiar tipo → **GPU L4**.
2. Ejecutar las celdas en orden. Si se cae la sesión, volver a ejecutarlas: cada corrida se reanuda desde su
   última época completa.
3. No se evalúa test ni se descargan pesos.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, os, json, hashlib, zipfile
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)
OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e3_semillas_v1/sesion_A')
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'
ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)
if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos y repite esta celda.')
def sha_file(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
assert sha_file(ZIP)==EXPECTED_ZIP_SHA256, 'ZIP incorrecto o modificado; usa el archivo v2 entregado con este notebook'
BASE=Path('/content/imo_e2_v2')
BASE.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    for n in z.namelist():
        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')
    z.extractall(BASE)
manifest=json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())
for name, expected in manifest['files'].items():
    assert sha_file(BASE/name)==expected, name
PROJECT=BASE/'tesis_imo_para_codex'
print('Paquete íntegro:',len(manifest['files']),'archivos')


## Entorno aislado y fijado
Se crea Python 3.11 fuera del kernel de Colab. La instalación puede tardar varios minutos; no cambia el Python del notebook. No se compilan extensiones CUDA: se usan ruedas publicadas por sus autores para torch 2.5/CUDA 12.


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)
ENV=Path('/content/imo-e2-env')
if not (ENV/'bin/python').exists():
    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)
PY=str(ENV/'bin/python')
def pip_install(*items):
    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)
pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')
pip_install('-r',str(PROJECT/'requirements-e2.txt'))
pip_install('transformers==4.46.3')
pip_install('--no-deps','https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/causal_conv1d-1.5.0.post8%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')
pip_install('--no-deps','https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')
subprocess.run([PY,'-c','import torch, torchvision, mamba_ssm; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)
freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)
(OUTPUT/'environment.freeze.txt').write_text(freeze)


## Comprobación previa obligatoria
Comprueba hashes de datos/código, separación de grupos, identidad arquitectónica, forward, gradientes, paso de optimizador y evaluación con datos sintéticos. Guarda memoria GPU y tiempo. Si falla un modelo, corrige el entorno o reduce **explícitamente** el alcance; no hay sustituciones automáticas. El lote nominal es 2; el último lote de una imagen se une al anterior (3) para BatchNorm.


In [ ]:
MODELS=['transunet_2d', 'pidnet_s']
subprocess.run([PY,str(PROJECT/'scripts/preflight_e2.py'),'--output',str(OUTPUT),'--models',*MODELS],check=True)
report=json.loads((OUTPUT/'preflight.json').read_text())
for name,result in report['models'].items(): print(name,result)


## Ejecutar o reanudar
Corre las redes de esta sesión con las cinco semillas nuevas. Se recupera la última época completa al reconectar.


In [ ]:
MODELS_RUN=MODELS
SEEDS=[89, 97, 101, 103, 107]
subprocess.run([PY,str(PROJECT/'scripts/experimento_e2_colab.py'),
    '--output',str(OUTPUT),'--models',*MODELS_RUN,'--seeds',*[str(s) for s in SEEDS]],check=True)


In [ ]:
for p in sorted(OUTPUT.glob('*/result.json')):
    r=json.loads(p.read_text())
    print(r['model'],r['seed'],r['completed'],r['best_val_group_Dice'],r['best_file'])
print('Los resultados permanecen en',OUTPUT)
